# 05 — Human Annotation Analysis

Analyses the three human evaluation datasets:
- **FC-4 Factual Precision**: 800 claims × 3 annotators, Fleiss κ = 0.76
- **TI-2 Explanation Faithfulness**: 500 explanations × 3 annotators, κ = 0.71
- **TI-3 User Interpretability**: 1,200 Likert ratings, Cronbach α = 0.84


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

fc4 = pd.read_csv('../data/annotations/FC4_factual_precision/fc4_all_systems_combined.csv')
ti2 = pd.read_csv('../data/annotations/TI2_explanation_faithfulness/ti2_all_systems_combined.csv')
ti3 = pd.read_csv('../data/annotations/TI3_user_interpretability/ti3_all_systems_combined.csv')

print(f"FC-4: {len(fc4)} rows  |  TI-2: {len(ti2)} rows  |  TI-3: {len(ti3)} rows")

FC-4: 800 rows  |  TI-2: 500 rows  |  TI-3: 1200 rows


## 1. FC-4 Factual Precision — label distribution by system

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Label distribution
label_counts = fc4.groupby(['system_id','majority_judgment']).size().unstack(fill_value=0)
label_counts_pct = label_counts.div(label_counts.sum(axis=1), axis=0) * 100
label_counts_pct.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71','#f39c12'],
                      edgecolor='white', linewidth=0.5)
axes[0].set_title('FC-4: Claim Label Distribution by System', fontsize=12)
axes[0].set_xlabel('System'); axes[0].set_ylabel('Percentage (%)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Judgment')
axes[0].grid(axis='y', alpha=0.3)

# Mean confidence by system
conf_mean = fc4.groupby('system_id')['mean_confidence'].mean()
conf_mean.plot(kind='bar', ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('FC-4: Mean Annotator Confidence by System', fontsize=12)
axes[1].set_xlabel('System'); axes[1].set_ylabel('Mean confidence (1–5)')
axes[1].set_ylim(0, 5); axes[1].tick_params(axis='x', rotation=0)
axes[1].axhline(y=4.0, color='grey', linestyle='--', linewidth=0.8, label='Target ≥4')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('FC-4 Factual Precision Annotation Analysis  (n=800, Fleiss κ=0.76)', fontsize=13)
plt.tight_layout()
plt.show()

print("\nFC-4 correct rate per system:")
print((label_counts_pct['CORRECT'].rename('correct_%')).round(1).to_string())


FC-4 correct rate per system:
S1    78.5
S2    83.8
S3    86.1
S4    80.2


## 2. TI-2 Explanation Faithfulness

In [ ]:
ti2_labels = ti2.groupby(['system_id','majority_label']).size().unstack(fill_value=0)
ti2_pct    = ti2_labels.div(ti2_labels.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
ti2_pct.plot(kind='bar', ax=ax,
             color=['#27ae60','#f39c12','#e74c3c'],
             edgecolor='white', linewidth=0.5)
ax.set_title('TI-2 Explanation Faithfulness Labels by System  (n=500, Fleiss κ=0.71)', fontsize=12)
ax.set_xlabel('System'); ax.set_ylabel('Percentage (%)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Label')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTI-2 faithful rate per system:")
for sys_id in sorted(ti2['system_id'].unique()):
    sub = ti2[ti2['system_id']==sys_id]
    faithful_pct = (sub['majority_label']=='FAITHFUL').mean() * 100
    print(f"  {sys_id}: {faithful_pct:.1f}% FAITHFUL")


TI-2 faithful rate per system:
  S1: 55.2% FAITHFUL
  S2: 59.6% FAITHFUL
  S3: 67.8% FAITHFUL
  S4: 72.4% FAITHFUL


## 3. TI-3 User Interpretability Ratings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of composite scores
for sys_id, color in zip(['S1','S3','S4'],['#4C72B0','#55A868','#C44E52']):
    sub = ti3[ti3['system_id']==sys_id]['composite_score']
    axes[0].hist(sub, bins=14, alpha=0.6, label=sys_id, color=color, edgecolor='white')
axes[0].axvline(3.8, color='black', linestyle='--', linewidth=1.2, label='Threshold (3.8)')
axes[0].set_title('TI-3: Composite Score Distribution  (n=1,200)', fontsize=12)
axes[0].set_xlabel('Composite score (1–5)'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].grid(alpha=0.2)

# Mean per question per system
q_cols = ['q1_clarity','q2_understandability','q3_trust','q4_usefulness','q5_satisfaction']
q_labels = ['Q1\nClarity','Q2\nUnderstand.','Q3\nTrust','Q4\nUsefulness','Q5\nSatisf.']
x = range(len(q_cols))
for sys_id, color in zip(['S1','S3','S4'],['#4C72B0','#55A868','#C44E52']):
    means = ti3[ti3['system_id']==sys_id][q_cols].mean()
    axes[1].plot(x, means, 'o-', label=sys_id, color=color, linewidth=2, markersize=7)
axes[1].axhline(3.8, color='grey', linestyle='--', linewidth=0.8, label='Threshold')
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(q_labels, fontsize=9)
axes[1].set_ylim(1, 5); axes[1].set_title('TI-3: Mean Score per Question', fontsize=12)
axes[1].legend(); axes[1].grid(alpha=0.2)

plt.suptitle('TI-3 User Interpretability Analysis  (Cronbach α=0.84, S2 excluded)', fontsize=13)
plt.tight_layout()
plt.show()

print("\nTI-3 composite means vs threshold (≥3.8):")
for sys_id in ['S1','S3','S4']:
    mean = ti3[ti3['system_id']==sys_id]['composite_score'].mean()
    status = '✓' if mean >= 3.8 else '✗'
    print(f"  {sys_id}: {mean:.2f} {status}")


TI-3 composite means vs threshold (≥3.8):
  S1: 2.84 ✗
  S3: 3.77 ✗
  S4: 4.48 ✓


## 4. IAA summary

In [ ]:
iaa = json.load(open('../interviews/codebook/iaa_coding_log.json'))
fc4_iaa = json.load(open('../data/annotations/FC4_factual_precision/fc4_iaa_summary.json'))
ti2_iaa = json.load(open('../data/annotations/TI2_explanation_faithfulness/ti2_iaa_summary.json'))
ti3_sum = json.load(open('../data/annotations/TI3_user_interpretability/ti3_survey_summary.json'))

print("=== Inter-Annotator Agreement Summary ===\n")
print(f"FC-4  Fleiss κ = {fc4_iaa['overall']['fleiss_kappa']:.2f}  (n={fc4_iaa['overall']['n_claims_total']} claims)")
print(f"TI-2  Fleiss κ = {ti2_iaa['overall_fleiss_kappa']:.2f}  (n={ti2_iaa['n_explanations']} explanations)")
print(f"TI-3  Cronbach α = {ti3_sum['cronbach_alpha']:.2f}  (n={ti3_sum['n_total_responses']} ratings)")
print(f"\nInterview coding  Cohen's κ = {iaa['overall_kappa']:.2f}  (n={iaa['n_sessions']} sessions)")
print("\nAll IAA values meet the paper's stated targets (κ≥0.70, α≥0.80).")

=== Inter-Annotator Agreement Summary ===

FC-4  Fleiss κ = 0.76  (n=800 claims)
TI-2  Fleiss κ = 0.71  (n=500 explanations)
TI-3  Cronbach α = 0.84  (n=1200 ratings)

Interview coding  Cohen's κ = 0.81  (n=12 sessions)

All IAA values meet the paper's stated targets (κ≥0.70, α≥0.80).
